# Notebook for manual implementation of Batch Norm

### Computational Graph
#### $ f_{1} = E[z_{i}] $
#### $ f_{2i} = z_{i} - f_{1} $

In [118]:
import numpy as np
import os
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import mnist1d
import random

In [119]:
def compute_f_1(z_i):
    result = np.mean(z_i)
    return result

In [120]:
def test_compute_f_1():
    input = np.array([ 2, 2, 2, 4, 4, 4 ])
    result = compute_f_1(input)
    assert(result == 3)

test_compute_f_1()

In [121]:
def compute_f_2i(z_i, f_1):
    result = z_i - f_1
    return result

In [122]:
def test_compute_f_2i():
    input = np.array([ 2, 2, 2, 4, 4, 4 ])
    result = compute_f_2i(input, compute_f_1(input))
    expected = np.array([2-3, 2-3, 2-3, 4-3, 4-3, 4-3])
    comparison = result == expected
    assert(comparison.all())

test_compute_f_2i()

### Computational Graph

#### We have the $ z_{i} $ now with the mean normalised; see `test_compute_f_2i()`

#### $ f_{3i} = f_{2i}^{2} $
#### $ f_{4} = E(f_{3i}) $
#### $ f_{5} = \sqrt{f_{4} + \epsilon} $
#### $ f_{6} = 1 / f_{5} $

In [123]:
def compute_f_3i(f_2):
    result = np.square(f_2)
    return result

def compute_f_4(f_3i):
    result = np.mean(f_3i)
    return result

def compute_f_5(f_4):
    assert(f_4 >= 0)
    epsilon = 1e-5
    result = np.sqrt(f_4 + epsilon)
    return result

def compute_f_6(f_5):
    result = 1 / f_5
    return result

In [124]:
def test_compute_f3i():
    input = np.array([2, 0, -1])
    result = compute_f_3i(input)
    expected = np.array([4, 0, 1])
    comparison = result == expected
    assert(comparison.all())

test_compute_f3i()

In [125]:
def test_compute_f_4():
    input = np.array([4, 0, 1])
    result = compute_f_4(input)
    assert((4+0+1)/3 == result)

test_compute_f_4()

In [126]:
def test_compute_f_5():
    assert(compute_f_5(0) > 0)
    result = compute_f_5(2)
    print(result)
    assert(1.41 < result)
    assert(result < 1.42)

test_compute_f_5()

1.4142170979025817


In [127]:
def test_compute_f_6():
    input = 0
    result1 = compute_f_5(input)
    assert(result1 > 0.003)
    result2 = compute_f_6(result1)
    assert(result2 > 316)
    assert(result2 < 317)

test_compute_f_6()
    

### Computational Graph

#### $ f_{7i} = f_{2i} * f_{6} $
#### $ z_{i}' = f_{7i} \times \gamma + \delta $

In [128]:
def compute_f_7i(f_2i, f_6):
    result = f_2i * f_6
    return result

def compute_z_i_prime(f_7i, gamma, delta):
    result = f_7i * gamma + delta
    return result

In [129]:
def test_compute_f_7i():
    result = compute_f_7i(2, 4)
    assert(result == 8)

def test_compute_z_i_prime():
    result = compute_z_i_prime(-0.2, 5, 3)
    assert(result == 2)

test_compute_f_7i()
test_compute_z_i_prime()

In [133]:
def test_overall():
    z_i = np.array([ 2, 2, 2, 4, 4, 4 ])
    gamma = 5
    delta = 2
    f_1 = compute_f_1(z_i)
    f_2i = compute_f_2i(z_i, f_1)
    f_3i = compute_f_3i(f_2i)
    f_4 = compute_f_4(f_3i)
    f_5 = compute_f_5(f_4)
    f_6 = compute_f_6(f_5)
    f_7i = compute_f_7i(f_2i, f_6)
    z_i_prime = compute_z_i_prime(f_7i, gamma, delta)
    print(z_i_prime)
    assert(z_i_prime[0] + 3 < 0.1)
    assert(z_i_prime[5] - 7 >-0.1)

test_overall()
    


[-2.999975 -2.999975 -2.999975  6.999975  6.999975  6.999975]
